ET-SSL Pre-Training Notebook
**Anomaly Detection in Encrypted Network Traffic using Self-Supervised Contrastive Learning**

Paper: s41598-025-08568-0 (Scientific Reports 2025)

Setup
Runtime: GPU (T4 or A100 recommended)
Estimated training time: ~45 min on T4 per dataset


## 1. Environment Setup — Clone Repo & Install Dependencies

In [ ]:
# ── 1a. Clone the Sentinel repository ────────────────────────────────────────
import os, sys

REPO_URL  = "https://github.com/YOUR_USERNAME/Sentinel.git"   # ← update this
REPO_DIR  = "Sentinel"
HYBRID_DIR = f"{REPO_DIR}/hybrid-detection"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL}")
else:
    os.system(f"git -C {REPO_DIR} pull --ff-only")   # keep up to date

# Add hybrid-detection to Python path so all config/model imports resolve
if HYBRID_DIR not in sys.path:
    sys.path.insert(0, HYBRID_DIR)

print("Repo ready. sys.path:", sys.path[:3])

In [ ]:
# ── 1b. Install dependencies ──────────────────────────────────────────────────
# (Colab already has torch/sklearn; optuna and joblib may need installing)
import subprocess
subprocess.run(["pip", "install", "-q", "optuna", "joblib", "tqdm"], check=True)
print("Dependencies installed.")

## 2. Imports from Sentinel Repo

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# ── Config ────────────────────────────────────────────────────────────────────
from config.constants import (
    FEATURE_NAMES, FEATURE_DIM,
    CONTINUOUS_INDICES, CONTINUOUS_MASK, CATEGORICAL_INDICES,
    BATCH_SIZE, LEARNING_RATE, LR_DECAY_FACTOR, LR_DECAY_EPOCHS,
    NUM_EPOCHS, TEMPERATURE_TAU, GAMMA, ALPHA_EMA,
    TRAIN_RATIO, VAL_RATIO, TEST_RATIO,
    EMBEDDING_DIM, PROJECTION_DIM,
)
from config.column_maps import (
    CIC_DARKNET2020_COLUMN_MAP,
    CIC_IDS2018_COLUMN_MAP,
    CICFLOWMETER_PROTO_MAP,
    UNSW_MAP,
    LABEL_NORMAL,
    LABEL_ANOMALY,
)

# ── Model components ──────────────────────────────────────────────────────────
from model.et_ssl      import ETSSLModel
from model.loss        import NTXentLoss, CombinedLoss
from model.preprocessor import Augmenter, TrafficScaler
from model.dataset     import TrafficDataset

# ── Feature builder (same fn used by the detection service) ──────────────────
from feature_extractor.feature_builder import build_feature_vector, build_feature_matrix

print(f"Feature dim: {FEATURE_DIM}")
print(f"Continuous indices: {CONTINUOUS_INDICES}")
print(f"Categorical indices: {CATEGORICAL_INDICES}")
print("All imports OK.")

## 3. Configuration

In [ ]:
DATASET_CHOICE = 'darknet'   # 'darknet' | 'ids2018' | 'unsw'
DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Paper hyperparameters — imported from constants, used as search defaults
DEFAULT_CONFIG = dict(
    lr          = LEARNING_RATE,       # 1e-3
    batch_size  = BATCH_SIZE,          # 256
    epochs      = NUM_EPOCHS,          # 100
    temperature = TEMPERATURE_TAU,     # 0.1
    gamma       = GAMMA,               # 0.5
    alpha_ema   = ALPHA_EMA,           # 0.95
    dropout     = 0.3,
    lr_decay    = LR_DECAY_FACTOR,     # 0.95
    lr_decay_ep = LR_DECAY_EPOCHS,     # 10
    embed_dim   = EMBEDDING_DIM,       # 64
    proj_dim    = PROJECTION_DIM,      # 32
    # Architecture (paper default)
    hidden_dims    = (128, 256, 128),
    n_hidden       = 3,
    hidden_w1      = 128,
    hidden_w2      = 256,
    hidden_w3      = 128,
    # Optimizer
    weight_decay   = 0.0,          # paper uses plain Adam (no decay)
    # Augmentation (paper default)
    noise_std      = 0.05,
    dropout_max_feat = 3,
    jitter_lo      = 0.9,
    jitter_hi      = 1.1,
)

# Mount Google Drive for dataset + artifact storage
from google.colab import drive
drive.mount('/content/drive')
DATASET_DIR = '/content/drive/MyDrive/Sentinel/datasets'
EXPORT_DIR  = '/content/drive/MyDrive/Sentinel/checkpoints'
import os; os.makedirs(EXPORT_DIR, exist_ok=True)
PLOTS_DIR   = f'{EXPORT_DIR}/plots/{DATASET_CHOICE}'
os.makedirs(PLOTS_DIR, exist_ok=True)
print(f"Dataset dir : {DATASET_DIR}")
print(f"Export  dir : {EXPORT_DIR}")
print(f"Plots   dir : {PLOTS_DIR}")

## 4. Data Loading & Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from feature_extractor.feature_builder import build_feature_vector

# Dataset configurations — column maps imported from config.column_maps
DATASET_CONFIGS = {
    'darknet': {
        'path':            f'{DATASET_DIR}/darknet2020.csv',
        'col_map':         CIC_DARKNET2020_COLUMN_MAP,
        'label_normal':    LABEL_NORMAL['darknet'],      # 'BENIGN'
        'fix_proto':       True,   # Protocol is numeric (6=TCP,17=UDP)
    },
    'ids2018': {
        'path':            f'{DATASET_DIR}/cic_ids2018.csv',
        'col_map':         CIC_IDS2018_COLUMN_MAP,
        'label_normal':    LABEL_NORMAL['ids2018'],      # 'Benign'
        'fix_proto':       True,   # Protocol is numeric (6=TCP,17=UDP)
    },
    'unsw': {
        'path':            f'{DATASET_DIR}/unsw_nb15.csv',
        'col_map':         UNSW_MAP,
        'label_normal':    LABEL_NORMAL['unsw'],         # 0
        'fix_proto':       False,  # proto is already a string ("tcp","udp")
    },
}

cfg = DATASET_CONFIGS[DATASET_CHOICE]
df  = pd.read_csv(cfg['path'], low_memory=False)

# Rename raw columns to unified feature names via the repo column map
df = df.rename(columns=cfg['col_map']).fillna(0)

# CICFlowMeter datasets store Protocol as an integer (6=TCP, 17=UDP).
# build_feature_vector() expects a string ("tcp"/"udp") — map it here.
if cfg['fix_proto']:
    df['proto'] = df['proto'].map(CICFLOWMETER_PROTO_MAP).fillna('other')

# Build feature matrix using build_feature_vector from feature_extractor/feature_builder.py
#
# Production pipeline (live traffic):
#   Kafka bytes → parse_kafka_message() [zeek_parser.py] → dict → build_feature_vector() → tensor
#
# Training pipeline (CSV datasets):
#   CSV row     → df.to_dict()                           → dict → build_feature_vector() → array
#
# Both paths feed the same dict schema into build_feature_vector(), guaranteeing
# identical feature layout between training and inference — no train/serve skew.
records = df.to_dict(orient='records')
X_all   = build_feature_matrix(records)

# ── Binary labeling — threat model decisions (see column_maps.py) ─────────────
# All three datasets use the simple rule: everything != label_normal is anomalous.
# Tor and VPN-C2 traffic in darknet are treated as anomalous (see column_maps.py).
y_all = (df['label'] != cfg['label_normal']).astype('int64').to_numpy()

print(f"Loaded {len(X_all):,} flows | anomaly rate: {y_all.mean():.2%}")
print(f"Feature matrix shape: {X_all.shape}  (expected N x {FEATURE_DIM})")

# Print class distribution to verify labeling
print(f"Loaded {len(X_all):,} flows | anomaly rate: {y_all.mean():.2%}")
print(f"  Normal:  {(y_all==0).sum():,}")
print(f"  Anomaly: {(y_all==1).sum():,}")
print(f"  Unique labels found: {df['label'].unique()[:10]}")

## 5. Train / Val / Test Split & Scaling

In [ ]:
# Split — ratios imported from constants (70 / 15 / 15)
val_frac_of_trainval = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)   # ≈ 0.1765

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_all, y_all, test_size=TEST_RATIO, stratify=y_all, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=val_frac_of_trainval,
    stratify=y_trainval, random_state=42
)
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

# Scale using TrafficScaler from model/preprocessor.py
# Fits only on CONTINUOUS_INDICES (0-11); one-hot columns (12-19) are untouched.
scaler = TrafficScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# Verify one-hot protection
assert np.array_equal(X_train_scaled[:, 12:], X_train[:, 12:]), "One-hot columns were modified!"
print("Scaler fitted. One-hot protection verified.")

## 6. Training Utilities

In [ ]:
# All classes (Augmenter, TrafficDataset, ETSSLModel, NTXentLoss) are already
# imported from the repo in Cell 2. This cell only defines the training loop.

@torch.no_grad()
def get_embeddings(model, X_np, device, batch_size=512):
    model.eval()
    dl = DataLoader(torch.from_numpy(X_np).float(), batch_size=batch_size)
    return torch.cat([model.encode(b.to(device)) for b in dl]).cpu().numpy()

def compute_auc(embeddings, centroid, labels):
    from sklearn.metrics import roc_auc_score
    scores = ((embeddings - centroid[None, :]) ** 2).sum(axis=1)
    try:    return roc_auc_score(labels, scores)
    except: return 0.5

def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total = 0.0
    for batch in loader:
        x_orig, x_aug = batch[0].to(device), batch[1].to(device)
        optimizer.zero_grad()
        _, h     = model(x_orig)
        _, h_aug = model(x_aug)
        loss = loss_fn(h, h_aug)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)

@torch.no_grad()
def eval_epoch(model, loader, loss_fn, device):
    model.eval()
    total = 0.0
    for batch in loader:
        x_orig, x_aug = batch[0].to(device), batch[1].to(device)
        _, h     = model(x_orig)
        _, h_aug = model(x_aug)
        total   += loss_fn(h, h_aug).item()
    return total / len(loader)

def run_training(config, X_train_s, X_val_s, y_val, device, trial_name='default'):
    aug    = Augmenter(
        noise_std=config.get('noise_std', 0.05),
        dropout_max=config.get('dropout_max_feat', 3),
        jitter_range=(config.get('jitter_lo', 0.9), config.get('jitter_hi', 1.1)),
    )
    ds     = TrafficDataset(X_train_s, augmenter=aug)
    dl     = DataLoader(ds, batch_size=config['batch_size'], shuffle=True,
                        num_workers=2, pin_memory=True, drop_last=True)
    val_ds = TrafficDataset(X_val_s, augmenter=aug)
    val_dl = DataLoader(val_ds, batch_size=512, shuffle=False, num_workers=2)

    model     = ETSSLModel(
                    hidden_dims=config.get('hidden_dims', (128, 256, 128)),
                    embed_dim=config['embed_dim'],
                    proj_dim=config['proj_dim'],
                    dropout=config['dropout'],
                ).to(device)
    # AdamW: Adam + decoupled weight decay (better for contrastive learning)
    # weight_decay=0.0 falls back to plain Adam
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config.get('weight_decay', 0.0),
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=config['lr_decay_ep'], gamma=config['lr_decay'])
    loss_fn   = NTXentLoss(temperature=config['temperature'])

    best_auc, best_state, best_centroid = 0.0, None, None
    history, last_centroid = [], None

    for epoch in range(1, config['epochs'] + 1):
        train_loss = train_one_epoch(model, dl, optimizer, loss_fn, device)
        val_loss   = eval_epoch(model, val_dl, loss_fn, device)
        scheduler.step()

        if epoch % 5 == 0 or epoch == 1:
            z_train  = get_embeddings(model, X_train_s, device)
            centroid = z_train.mean(axis=0)
            if last_centroid is not None:
                centroid = (config['alpha_ema'] * last_centroid
                            + (1 - config['alpha_ema']) * centroid)
            last_centroid = centroid

            z_val = get_embeddings(model, X_val_s, device)
            auc   = compute_auc(z_val, centroid, y_val)
            history.append({'epoch': epoch, 'train_loss': train_loss,
                            'val_loss': val_loss, 'val_auc': auc})
            print(f"[{trial_name}] Ep {epoch:3d} | "
                  f"train={train_loss:.4f} val={val_loss:.4f} AUC={auc:.4f}")
            if auc > best_auc:
                best_auc      = auc
                best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                best_centroid = centroid.copy()
        else:
            history.append({'epoch': epoch, 'train_loss': train_loss,
                            'val_loss': val_loss})

    model.load_state_dict(best_state)
    return model, best_centroid, best_auc, history

9. Quick Baseline Training (Paper Hyperparameters)


In [ ]:
baseline_model, baseline_centroid, baseline_auc, baseline_history = run_training(
    DEFAULT_CONFIG, X_train_scaled, X_val_scaled, y_val, DEVICE, trial_name='baseline'
)
print(f"\nBaseline best val AUC: {baseline_auc:.4f}")


10. Hyperparameter Search with Optuna


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    # ── Learning dynamics ────────────────────────────────────────────────────
    config = dict(
        lr          = trial.suggest_float('lr',         1e-4, 3e-3, log=True),
        batch_size  = trial.suggest_categorical('batch_size',  [128, 256, 512]),
        epochs      = 60,   # reduced for search speed; final run uses 150
        temperature = trial.suggest_categorical('temperature', [0.05, 0.1, 0.2, 0.5]),
        gamma       = trial.suggest_float('gamma',      0.1, 1.0),
        alpha_ema   = trial.suggest_categorical('alpha_ema',   [0.9, 0.95, 0.99]),
        dropout     = trial.suggest_float('dropout',    0.1, 0.5),
        lr_decay    = trial.suggest_categorical('lr_decay',    [0.9, 0.95, 0.98]),
        lr_decay_ep = trial.suggest_categorical('lr_decay_ep', [5, 10, 20]),

        # ── Architecture: depth and width ────────────────────────────────────
        # n_hidden controls how many hidden blocks the encoder has (2 or 3).
        # hidden_w1/w2/w3 are the widths of each block.
        # Wider middle layers let the encoder expand representations before
        # compressing to embed_dim; the search finds the right tradeoff.
        n_hidden    = trial.suggest_categorical('n_hidden', [2, 3]),
        hidden_w1   = trial.suggest_categorical('hidden_w1', [64, 128, 256]),
        hidden_w2   = trial.suggest_categorical('hidden_w2', [128, 256, 512]),
        hidden_w3   = trial.suggest_categorical('hidden_w3', [64, 128, 256]),
        embed_dim   = trial.suggest_categorical('embed_dim', [32, 64, 128]),
        proj_dim    = trial.suggest_categorical('proj_dim',  [16, 32, 64]),

        # ── Augmentation quality ─────────────────────────────────────────────
        # Gaussian noise: too small → trivial views; too large → encoder can't
        # pull augmented pairs together → high loss, poor representations.
        noise_std       = trial.suggest_float('noise_std',   0.01, 0.2,  log=True),
        # Feature dropout: randomly zeros out 1–dropout_max continuous columns.
        dropout_max_feat = trial.suggest_int('dropout_max_feat', 1, 5),
        # Scale jitter: multiplicative noise on continuous features.
        jitter_lo       = trial.suggest_float('jitter_lo', 0.80, 0.95),
        jitter_hi       = trial.suggest_float('jitter_hi', 1.05, 1.30),

        # ── Optimizer ────────────────────────────────────────────────────────
        # AdamW adds decoupled weight decay to Adam — often better in contrastive
        # settings as it prevents embedding collapse. weight_decay=0 reduces to Adam.
        weight_decay    = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True),
    )

    # Build hidden_dims tuple from sampled width + depth
    if config['n_hidden'] == 2:
        config['hidden_dims'] = (config['hidden_w1'], config['hidden_w2'])
    else:  # 3 hidden layers (paper default)
        config['hidden_dims'] = (config['hidden_w1'], config['hidden_w2'], config['hidden_w3'])

    _, _, auc, _ = run_training(config, X_train_scaled, X_val_scaled, y_val,
                                 DEVICE, trial_name=f"trial-{trial.number}")
    return auc

N_TRIALS = 50   # 50 gives good coverage; use 100 for thorough search
study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=42))

# Seed with paper hyperparameters as first trial (ensures paper baseline is covered)
paper_trial = {**DEFAULT_CONFIG,
               'n_hidden': 3, 'hidden_w1': 128, 'hidden_w2': 256, 'hidden_w3': 128,
               'noise_std': 0.05, 'dropout_max_feat': 3,
               'jitter_lo': 0.9, 'jitter_hi': 1.1}
study.enqueue_trial(paper_trial)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("\n=== Best Trial ===")
print(f"Best AUC: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")
print(f"Best architecture: {study.best_params.get('n_hidden')} hidden layers, "
      f"dims={study.best_params.get('hidden_w1')}/{study.best_params.get('hidden_w2')}"
      + (f"/{study.best_params.get('hidden_w3')}" if study.best_params.get('n_hidden')==3 else ""))
print(f"Best augmentation: noise_std={study.best_params.get('noise_std'):.4f}, "
      f"dropout_max={study.best_params.get('dropout_max_feat')}, "
      f"jitter=[{study.best_params.get('jitter_lo'):.2f},{study.best_params.get('jitter_hi'):.2f}]")

## 11a. Hyperparameter Analysis Plots
Visualising the Optuna study results to understand the effect of each parameter group on model AUC.
Used for methodology discussion.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd

# ── Build a DataFrame of all Optuna trials ────────────────────────────────────
records = []
for t in study.trials:
    if t.value is not None:
        row = {'auc': t.value, **t.params}
        # Reconstruct hidden_dims label for display
        n = t.params.get('n_hidden', 3)
        w1 = t.params.get('hidden_w1', 128)
        w2 = t.params.get('hidden_w2', 256)
        w3 = t.params.get('hidden_w3', 128)
        row['arch_label'] = f"{n}L: {w1}→{w2}" + (f"→{w3}" if n == 3 else "")
        records.append(row)

df_trials = pd.DataFrame(records)
print(f"Total completed trials: {len(df_trials)}")
print(df_trials[['auc', 'n_hidden', 'hidden_w1', 'hidden_w2', 'temperature',
                  'noise_std', 'dropout_max_feat', 'weight_decay']].describe().round(4))

# ── Colour map: green=high AUC, red=low AUC ──────────────────────────────────
auc_norm = (df_trials['auc'] - df_trials['auc'].min()) / (df_trials['auc'].max() - df_trials['auc'].min() + 1e-9)
colors = plt.cm.RdYlGn(auc_norm)

fig = plt.figure(figsize=(20, 24))
fig.suptitle(f'Hyperparameter Analysis — ET-SSL ({DATASET_CHOICE})', fontsize=16, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

def scatter_param(ax, param, xlabel, log=False):
    if param not in df_trials.columns:
        ax.set_visible(False); return
    ax.scatter(df_trials[param], df_trials['auc'], c=colors, alpha=0.75, edgecolors='k', linewidths=0.4, s=60)
    if log:
        ax.set_xscale('log')
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('Val AUC', fontsize=10)
    # Best value marker
    best_idx = df_trials['auc'].idxmax()
    ax.scatter(df_trials.loc[best_idx, param], df_trials.loc[best_idx, 'auc'],
               marker='*', s=250, color='gold', edgecolors='black', zorder=5, label='Best')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

def box_param(ax, param, xlabel):
    if param not in df_trials.columns:
        ax.set_visible(False); return
    groups = df_trials.groupby(param)['auc'].apply(list)
    ax.boxplot(groups.values, labels=[str(k) for k in groups.index], patch_artist=True,
               boxprops=dict(facecolor='#4C72B0', alpha=0.6))
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('Val AUC', fontsize=10)
    ax.grid(True, alpha=0.3)

# Row 1 — Augmentation parameters
ax1 = fig.add_subplot(gs[0, 0])
scatter_param(ax1, 'noise_std', 'Gaussian Noise σ (noise_std)', log=True)
ax1.set_title('Augmentation: Noise Strength', fontweight='bold')

ax2 = fig.add_subplot(gs[0, 1])
box_param(ax2, 'dropout_max_feat', 'Max Features Dropped')
ax2.set_title('Augmentation: Feature Dropout', fontweight='bold')

ax3 = fig.add_subplot(gs[0, 2])
if 'jitter_lo' in df_trials.columns and 'jitter_hi' in df_trials.columns:
    jitter_range = df_trials['jitter_hi'] - df_trials['jitter_lo']
    ax3.scatter(jitter_range, df_trials['auc'], c=colors, alpha=0.75, edgecolors='k', linewidths=0.4, s=60)
    ax3.set_xlabel('Jitter Range (hi - lo)', fontsize=10)
    ax3.set_ylabel('Val AUC', fontsize=10)
    ax3.grid(True, alpha=0.3)
ax3.set_title('Augmentation: Scale Jitter Range', fontweight='bold')

# Row 2 — Architecture
ax4 = fig.add_subplot(gs[1, 0])
box_param(ax4, 'n_hidden', 'Number of Hidden Layers')
ax4.set_title('Architecture: Depth', fontweight='bold')

ax5 = fig.add_subplot(gs[1, 1])
box_param(ax5, 'hidden_w1', 'First Hidden Width')
ax5.set_title('Architecture: Layer 1 Width', fontweight='bold')

ax6 = fig.add_subplot(gs[1, 2])
box_param(ax6, 'embed_dim', 'Embedding Dimension')
ax6.set_title('Architecture: Embed Dim', fontweight='bold')

# Row 3 — Contrastive loss parameters
ax7 = fig.add_subplot(gs[2, 0])
box_param(ax7, 'temperature', 'Temperature τ')
ax7.set_title('Loss: NT-Xent Temperature', fontweight='bold')
# Annotation: low τ → sharper distribution (harder negatives)
ax7.text(0.02, 0.97, 'Low τ → sharper, harder negatives\nHigh τ → softer distribution',
         transform=ax7.transAxes, fontsize=7, va='top', color='#444')

ax8 = fig.add_subplot(gs[2, 1])
scatter_param(ax8, 'lr', 'Learning Rate', log=True)
ax8.set_title('Optimizer: Learning Rate', fontweight='bold')

ax9 = fig.add_subplot(gs[2, 2])
scatter_param(ax9, 'weight_decay', 'Weight Decay (AdamW)', log=True)
ax9.set_title('Optimizer: Weight Decay', fontweight='bold')

# Row 4 — Optimization history + parameter importance
ax10 = fig.add_subplot(gs[3, 0:2])
# Optimization history: AUC over trial number
aucs_sorted = [t.value for t in study.trials if t.value is not None]
best_so_far = [max(aucs_sorted[:i+1]) for i in range(len(aucs_sorted))]
ax10.plot(range(len(aucs_sorted)), aucs_sorted, 'o', alpha=0.5, markersize=4, color='steelblue', label='Trial AUC')
ax10.plot(range(len(best_so_far)), best_so_far, '-', color='darkorange', linewidth=2, label='Best so far')
ax10.axhline(baseline_auc, color='red', linestyle='--', label=f'Paper baseline ({baseline_auc:.4f})')
ax10.set_xlabel('Trial Number', fontsize=10)
ax10.set_ylabel('Val AUC', fontsize=10)
ax10.set_title('Optuna Optimization History', fontweight='bold')
ax10.legend(fontsize=9)
ax10.grid(True, alpha=0.3)

ax11 = fig.add_subplot(gs[3, 2])
# Parameter importance via Optuna
try:
    importances = optuna.importance.get_param_importances(study)
    params_imp  = list(importances.keys())[:10]
    values_imp  = [importances[p] for p in params_imp]
    ax11.barh(params_imp[::-1], values_imp[::-1], color='#4C72B0', alpha=0.8)
    ax11.set_xlabel('Importance (fANOVA)', fontsize=10)
    ax11.set_title('Parameter Importance', fontweight='bold')
    ax11.grid(True, alpha=0.3, axis='x')
except Exception as e:
    ax11.text(0.5, 0.5, f'Importance requires\n≥20 trials\n({e})',
              ha='center', va='center', transform=ax11.transAxes)

plt.savefig(f'{PLOTS_DIR}/hyperparam_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {PLOTS_DIR}/hyperparam_analysis.png")


11. Final Training with Best Hyperparameters


In [ ]:
best_config = {**DEFAULT_CONFIG, **study.best_params}
best_config['epochs'] = 150   # Full training with best params

print("Training final model with best hyperparameters...")
final_model, final_centroid, final_auc, final_history = run_training(
    best_config, X_train_scaled, X_val_scaled, y_val, DEVICE, trial_name='final'
)
print(f"Final best val AUC: {final_auc:.4f}")


12. Threshold Calibration & Test Evaluation


In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, classification_report
import json

# Calibrate threshold on validation set
z_val      = get_embeddings(final_model, X_val_scaled, DEVICE)
val_scores = ((z_val - final_centroid[None,:])**2).sum(axis=1)

thresholds = np.linspace(val_scores.min(), val_scores.max(), 300)
best_f1, best_thresh = 0.0, thresholds[0]
for t in thresholds:
    preds = (val_scores > t).astype(int)
    f1 = f1_score(y_val, preds, zero_division=0)
    if f1 > best_f1:
        best_f1, best_thresh = f1, t

print(f"Calibrated threshold: {best_thresh:.6f}  (val F1: {best_f1:.4f})")

# Evaluate on test set
z_test      = get_embeddings(final_model, X_test_scaled, DEVICE)
test_scores = ((z_test - final_centroid[None,:])**2).sum(axis=1)
test_preds  = (test_scores > best_thresh).astype(int)
test_auc    = roc_auc_score(y_test, test_scores)

print("\n=== Test Set Results ===")
print(f"AUC:   {test_auc:.4f}")
print(classification_report(y_test, test_preds, target_names=['Normal','Anomaly']))


13. Export Artifacts to Drive


In [ ]:
import torch, json, joblib

# Save encoder weights
torch.save(final_model.state_dict(),
           f'{EXPORT_DIR}/encoder_{DATASET_CHOICE}.pt')

# Save scaler
joblib.dump(scaler, f'{EXPORT_DIR}/scaler_{DATASET_CHOICE}.joblib')

# Save centroid
np.save(f'{EXPORT_DIR}/centroid_{DATASET_CHOICE}.npy', final_centroid)

# Save threshold + config
meta = {
    'dataset':    DATASET_CHOICE,
    'threshold':  float(best_thresh),
    'val_auc':    float(final_auc),
    'test_auc':   float(test_auc),
    'best_config': best_config,
    'feature_dim': FEATURE_DIM,
    'embed_dim':   best_config['embed_dim'],
    'proj_dim':    best_config['proj_dim'],
}
with open(f'{EXPORT_DIR}/meta_{DATASET_CHOICE}.json', 'w') as f:
    json.dump(meta, f, indent=2)

print("Artifacts saved to Drive:")
print(f"  encoder_{DATASET_CHOICE}.pt")
print(f"  scaler_{DATASET_CHOICE}.joblib")
print(f"  centroid_{DATASET_CHOICE}.npy")
print(f"  meta_{DATASET_CHOICE}.json")


14. Training Curve Visualization


In [ ]:
import matplotlib.pyplot as plt

epochs_plot = [h['epoch'] for h in final_history if 'val_auc' in h]
aucs_plot   = [h['val_auc'] for h in final_history if 'val_auc' in h]
train_losses = [h['train_loss'] for h in final_history]
epochs_all   = [h['epoch'] for h in final_history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs_all, train_losses, label='Train Loss', color='steelblue')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('NT-Xent Loss')
axes[0].set_title('Training Loss'); axes[0].legend()

axes[1].plot(epochs_plot, aucs_plot, marker='o', color='darkorange', label='Val AUC')
axes[1].axhline(test_auc, color='red', linestyle='--', label=f'Test AUC={test_auc:.4f}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('Validation AUC'); axes[1].legend()

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/training_curves.png', dpi=150)
plt.show()
print("Training complete!")